# Cellpose fine-tuning on EGFP-GRAM-W (Colab)

Runtime: **CPU** is enough for Sections 1–3. Use **T4 GPU** for Sections 4–5.
After any reset: run Section 0, then **Runtime → Run before** on the cell you want to continue from.

**Folders on Drive (`cellpose_GRAMW/`)**
- `anno_all_updated/` — all images + `<name>_ROI.zip`, with abnormal cells tagged `abn_…` and uncertain ones `unsure_…`
- `split.csv` — permanent record: train / test for every image
- `roi_tags.csv` — every ROI's name and tag (normal / abnormal / unsure), from the `abn_` / `unsure_` name prefixes
- `data_0.26/train`, `data_0.26/test` — training data at the standard pixel size (one folder per `TARGET`)
- `models/` — trained models; `<name>.json` next to each records the `TARGET` it was trained at

## 0. Setup (run after every reset)

In [ ]:
!pip install -q "cellpose>=4,<5" roifile

In [ ]:
# ===== SETUP =====
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import torch
from skimage.transform import rescale, resize
from skimage.draw import polygon
from skimage.segmentation import clear_border, relabel_sequential
from roifile import roiread
from cellpose import models, utils, train, metrics, io
io.logger_setup()                       # show Cellpose progress messages

# ---- settings ----
GRAM           = 0      # FIJI channel 1 = Python index 0
TARGET         = 0.26   # standard pixel size (µm/pixel) used for training and segmentation
SR_MAX         = 0.09   # pixel size (µm) below which an image counts as SR
EDGE_BUFFER_UM = 0.8    # cells closer than this to the image edge are removed

# ---- folders ----
folder      = Path("/content/drive/MyDrive/cellpose_GRAMW")
ANNOT_DIRS  = [folder / "anno_all_updated"]   # add future batches here
SPLIT_FILE  = folder / "split.csv"
TAGS_FILE   = folder / "roi_tags.csv"
model_dir   = folder / "models"
data_dir    = folder / f"data_{TARGET:.2f}"
train_dir   = data_dir / "train"
test_dir    = data_dir / "test"
for d in (model_dir, train_dir, test_dir):
    d.mkdir(parents=True, exist_ok=True)

# ---- loading images ----
def pixel_size(path):
    """Return the pixel size in µm, or None if the file doesn't record it."""
    with tifffile.TiffFile(path) as tif:
        xres = tif.pages[0].tags.get("XResolution")
    if xres is None:
        return None
    pixels, length = xres.value
    return length / pixels

def is_sr(path):
    return pixel_size(path) < SR_MAX

def get_gram(img):
    """Return the GRAM-W channel as a 2D image, whatever the file layout."""
    if img.ndim == 2:            # single-channel file: the whole image is GRAM-W
        return img
    if img.ndim == 3:            # (channels, y, x)
        return img[GRAM]
    raise ValueError(f"Unexpected image shape {img.shape}")

def load_gram_standard(path, target=TARGET):
    """Load the GRAM-W channel and resample it to the given pixel size."""
    img = tifffile.imread(path)
    scale = pixel_size(path) / target
    return rescale(get_gram(img), scale, order=1, preserve_range=True, anti_aliasing=True)

def find_pairs(dirs):
    """Every image in the given folders with its <name>_ROI.zip (or .roi) file."""
    pairs = []
    for d in dirs:
        for f in sorted(d.glob("*.tif*")):
            roi = None
            for ext in (".zip", ".roi"):
                candidate = f.with_name(f.stem + "_ROI" + ext)
                if candidate.exists():
                    roi = candidate
                    break
            if roi:
                pairs.append((f, roi))
            else:
                print("no ROI file for", d.name, "/", f.name)
    return pairs

def load_set(d):
    """Load every image/mask pair from a train or test folder."""
    X, Y = [], []
    for mp in sorted(d.glob("*_masks.tif")):
        X.append(tifffile.imread(mp.with_name(mp.name.replace("_masks", ""))))
        Y.append(tifffile.imread(mp).astype(np.int32))
    return X, Y

# ---- ROIs and masks ----
def read_rois(path):
    """Always return a list, whether the file holds one ROI (.roi) or many (.zip)."""
    rois = roiread(path)
    return rois if isinstance(rois, list) else [rois]

def rois_to_mask(rois, shape):
    """Paint ROIs into a label image: 0 = background, 1, 2, 3... = cells (in ROI order)."""
    mask = np.zeros(shape, dtype=np.int32)
    for n, roi in enumerate(rois, start=1):
        xy = roi.coordinates()
        rr, cc = polygon(xy[:, 1] - 0.5, xy[:, 0] - 0.5, shape)
        mask[rr, cc] = n
    return mask

def roi_tag(name):
    """Tag from the ROI name prefix: abn_ → abnormal, unsure_ → unsure, anything else → normal."""
    name = (name or "").lower()
    if name.startswith("abn_"):
        return "abnormal"
    if name.startswith("unsure_"):
        return "unsure"
    return "normal"

def remove_edge(m, buffer_px):
    """Drop cells within buffer_px pixels of the image border; renumber the rest 1, 2, 3..."""
    return relabel_sequential(clear_border(m, buffer_size=buffer_px))[0]

def edge_buffer_px(pixel_um):
    """EDGE_BUFFER_UM converted to pixels at a given pixel size (at least 1 pixel)."""
    return max(1, round(EDGE_BUFFER_UM / pixel_um))

def false_positives(pred, truth, thr=0.5):
    """Return the labels of predicted cells that don't match any true cell at IoU >= thr."""
    fps = []
    for lab in range(1, pred.max() + 1):
        cell = pred == lab
        if not cell.any():
            continue
        under = truth[cell]
        under = under[under > 0]
        if under.size == 0:
            fps.append(lab)
            continue
        best = np.bincount(under).argmax()
        t = truth == best
        iou = (cell & t).sum() / (cell | t).sum()
        if iou < thr:
            fps.append(lab)
    return fps

# ---- models ----
def load_model(name):
    """'pretrained' or the name of a model in models/."""
    if name == "pretrained":
        return models.CellposeModel(gpu=True)
    path = model_dir / name
    if not path.exists():
        raise FileNotFoundError(f"No model file {path}")
    return models.CellposeModel(gpu=True, pretrained_model=str(path))

def model_target(name):
    """The TARGET a model was trained at (from its .json; 0.26 if there is none)."""
    info = model_dir / f"{name}.json"
    return json.loads(info.read_text())["TARGET"] if info.exists() else 0.26

def predict_full(m, path, target):
    """Segment one image at `target` µm/pixel and return the masks at the original image size."""
    gram_full = get_gram(tifffile.imread(path))
    small = rescale(gram_full, pixel_size(path) / target, order=1, preserve_range=True, anti_aliasing=True)
    pred = m.eval(small)[0]
    return resize(pred, gram_full.shape, order=0, preserve_range=True, anti_aliasing=False).astype(np.int32)

# ---- display ----
def show_outlines(image, masks, ax, color="red", title=""):
    """Draw an image with the outline of every labelled cell on top."""
    lo, hi = np.percentile(image, [1, 99.5])
    ax.imshow(image, cmap="gray", vmin=lo, vmax=hi)
    for o in utils.outlines_list(masks):
        ax.plot(o[:, 0], o[:, 1], color=color, linewidth=1)
    ax.set_title(title)
    ax.axis("off")

print("setup done | TARGET =", TARGET, "| data folder:", data_dir.name)

## 1. Find and check the annotated images

Every image should have 2 or 3 numbers in its shape and a pixel size.

In [ ]:
pairs = find_pairs(ANNOT_DIRS)
originals = {f.stem: f for f, _ in pairs}
roi_of    = {f.stem: r for f, r in pairs}

stems = [f.stem for f, _ in pairs]
dups = {s for s in stems if stems.count(s) > 1}
if dups:
    raise ValueError(f"Same file name in two folders (would overwrite each other): {dups}")

for f, _ in pairs:
    with tifffile.TiffFile(f) as tif:
        shape = tif.series[0].shape
    px = pixel_size(f)
    warn = "  ⚠ CHECK" if px is None or len(shape) not in (2, 3) else ""
    print(f"{f.parent.name:17s} {str(shape):18s} {px if px is None else round(px, 4)}  {f.name}{warn}")
print(len(pairs), "image/ROI pairs")

## 2. Train / test assignment (from `split.csv`)

- Images already in `split.csv` keep their assignment, so the test set stays the same as for v2/v3 (names are unchanged).
- Any new image added later gets a stratified split: about `NEW_TEST_FRACTION` goes to test, picked separately from SR and non-SR.

In [ ]:
SEED = 0
NEW_TEST_FRACTION = 0.25

def stratified_pick(files, n_test, sr_frac, seed):
    """Pick test images separately from SR and non-SR so both are represented."""
    rng = random.Random(seed)
    sr  = [f.stem for f in files if is_sr(f)]
    nsr = [f.stem for f in files if not is_sr(f)]
    rng.shuffle(sr)
    rng.shuffle(nsr)
    n_sr = min(len(sr), max(1, round(len(sr) * sr_frac))) if sr else 0
    return set(sr[:n_sr] + nsr[:n_test - n_sr])

if SPLIT_FILE.exists():
    split = pd.read_csv(SPLIT_FILE)[["name", "split"]]
    print("loaded", SPLIT_FILE.name)
else:
    split = pd.DataFrame({"name": [], "split": []})
    print("⚠ split.csv not found: creating a NEW split. Scores will not be comparable with earlier models.")

new = [f for f, _ in pairs if f.stem not in set(split["name"])]
if new:
    n_new_test = round(len(new) * NEW_TEST_FRACTION)
    test_new = stratified_pick(new, n_new_test, NEW_TEST_FRACTION, SEED)
    added = pd.DataFrame({"name":  [f.stem for f in new],
                          "split": ["test" if f.stem in test_new else "train" for f in new]})
    split = added if split.empty else pd.concat([split, added], ignore_index=True)
    print(f"assigned {len(new)} new images: {len(test_new)} test, {len(new) - len(test_new)} train")
else:
    print("no new images")

missing = set(split["name"]) - set(originals)
if missing:
    print("⚠ in split.csv but not found in ANNOT_DIRS (ignored):", missing)
split = split[split["name"].isin(originals)].reset_index(drop=True)
split["mode"] = ["SR" if is_sr(originals[n]) else "non-SR" for n in split["name"]]
split.to_csv(SPLIT_FILE, index=False)
split.groupby(["split", "mode"]).size()

## 3. Build the training data at `TARGET`, and the ROI tag table

All ROIs (normal, `abn_`, `unsure_`) go into the masks: segmentation learns to find every cell.
Their tags are saved separately in `roi_tags.csv` for later filtering/classification.

In [ ]:
for d in (train_dir, test_dir):
    for p in d.glob("*.tif"):
        p.unlink()

where = dict(zip(split["name"], split["split"]))
n_cells = {"train": 0, "test": 0}
tag_rows = []
for f, roi_path in pairs:
    rois = read_rois(roi_path)
    for n, roi in enumerate(rois, start=1):
        tag_rows.append({"image": f.stem, "split": where[f.stem], "label": n,
                         "roi_name": roi.name, "tag": roi_tag(roi.name)})
    print("▶", f.name, flush=True)
    gram_full = get_gram(tifffile.imread(f))
    mask_full = rois_to_mask(rois, gram_full.shape)
    gram = load_gram_standard(f)
    mask = resize(mask_full, gram.shape, order=0, preserve_range=True, anti_aliasing=False)

    s = where[f.stem]
    dest = test_dir if s == "test" else train_dir
    tifffile.imwrite(dest / f"{f.stem}.tif", gram.astype(np.float32))
    tifffile.imwrite(dest / f"{f.stem}_masks.tif", mask.astype(np.uint16))
    n_cells[s] += len(rois)

tags = pd.DataFrame(tag_rows)
tags.to_csv(TAGS_FILE, index=False)
print(n_cells)
tags.groupby(["split", "tag"]).size()

In [ ]:
mask_path = sorted(train_dir.glob("*_masks.tif"))[-1]
X = tifffile.imread(mask_path.with_name(mask_path.name.replace("_masks", "")))
Y = tifffile.imread(mask_path)
fig, ax = plt.subplots(figsize=(8, 8))
show_outlines(X, Y, ax, color="lime", title=mask_path.name)
plt.show()

## 4. Train (GPU)

- Change `MODEL_NAME` for every new version, otherwise the old model is overwritten.
- Checkpoints are saved every 5 epochs. If the session dies: set `RESUME = True`, set `N_EPOCHS` to the epochs still missing, run again.
- `models/<MODEL_NAME>.json` records the `TARGET` and settings, so evaluation uses the right pixel size.

In [ ]:
MODEL_NAME    = "GRAMW_v4"
N_EPOCHS      = 25
LEARNING_RATE = 1e-5
RESUME        = False

X_train, Y_train = load_set(train_dir)
X_test,  Y_test  = load_set(test_dir)
print(len(X_train), "train images,", len(X_test), "test images at TARGET", TARGET)

ft_model = load_model(MODEL_NAME) if RESUME else models.CellposeModel(gpu=True)
model_path, train_losses, test_losses = train.train_seg(
    ft_model.net,
    train_data=X_train, train_labels=Y_train,
    test_data=X_test,   test_labels=Y_test,
    learning_rate=LEARNING_RATE, weight_decay=0.1,
    n_epochs=N_EPOCHS, batch_size=1, min_train_masks=1,
    save_every=5,
    save_path=folder, model_name=MODEL_NAME,
)
(model_dir / f"{MODEL_NAME}.json").write_text(json.dumps(
    {"TARGET": TARGET, "epochs": N_EPOCHS, "learning_rate": LEARNING_RATE, "n_train": len(X_train)}))
print("saved to", model_path)

In [ ]:
tl = np.array(test_losses)
ep = np.nonzero(tl)[0]
plt.plot(train_losses, label="train")
plt.plot(ep, tl[ep], "o-", label="test")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

## 5. Evaluate on the test set (GPU)

Scored at **full resolution** against your ROIs, with edge cells removed (`EDGE_BUFFER_UM`), as the final pipeline will run.
Each model segments at the `TARGET` it was trained at. Two scores:

- **all cells:** every ROI (normal, `abn_`, `unsure_`) counts. Does the model find every cell?
- **normal cells:** only normal ROIs count; detections of `abn_`/`unsure_` cells are ignored rather than counted as errors. How good is it on the cells you analyse?

In [ ]:
def drop_on(mask_pred, region):
    """Remove predicted cells that lie mostly (>50%) inside `region` (a True/False image)."""
    out = mask_pred.copy()
    for lab in range(1, mask_pred.max() + 1):
        cell = mask_pred == lab
        if cell.any() and region[cell].mean() > 0.5:
            out[cell] = 0
    return relabel_sequential(out)[0]

def evaluate(model_names, show=False):
    available = [n for n in model_names if n == "pretrained" or (model_dir / n).exists()]
    for n in set(model_names) - set(available):
        print(f"⚠ skipping {n}: no file in models/")

    names = list(split.loc[split["split"] == "test", "name"])
    files = [originals[n] for n in names]
    modes = ["SR" if is_sr(f) else "non-SR" for f in files]

    truth_all, truth_normal, ignore, images = [], [], [], []
    for f in files:
        gram_full = get_gram(tifffile.imread(f))
        rois = read_rois(roi_of[f.stem])
        full = rois_to_mask(rois, gram_full.shape)
        full = clear_border(full, buffer_size=edge_buffer_px(pixel_size(f)))   # labels keep their ROI numbers
        normal_labels = [n for n, r in enumerate(rois, start=1) if roi_tag(r.name) == "normal"]
        is_normal = np.isin(full, normal_labels)
        truth_all.append(relabel_sequential(full)[0])
        truth_normal.append(relabel_sequential(np.where(is_normal, full, 0))[0])
        ignore.append((full > 0) & ~is_normal)          # pixels of abn_/unsure_ cells
        images.append(gram_full)

    results = {}
    for name in available:
        m, t = load_model(name), model_target(name)
        print(f"segmenting with {name} at {t} µm/pixel…", flush=True)
        results[name] = [remove_edge(predict_full(m, f, t), edge_buffer_px(pixel_size(f))) for f in files]
        del m
        torch.cuda.empty_cache()

    def score_line(truth, preds):
        ap, tp, fp, fn = metrics.average_precision(truth, preds, threshold=[0.5, 0.75, 0.9])
        a = np.nanmean(ap, axis=0)
        return (f"AP@0.5={a[0]:.3f}  AP@0.75={a[1]:.3f}  AP@0.9={a[2]:.3f}"
                f"  | TP={tp.sum(0)[0]:.0f}  FP={fp.sum(0)[0]:.0f}  FN={fn.sum(0)[0]:.0f}")

    for kind in ["all cells", "normal cells"]:
        for group in ["all", "SR", "non-SR"]:
            idx = [k for k, md_ in enumerate(modes) if group == "all" or md_ == group]
            print(f"--- {kind} | {group} images ({len(idx)}) ---")
            for name, preds in results.items():
                if kind == "all cells":
                    t_ = [truth_all[k] for k in idx]
                    p_ = [preds[k] for k in idx]
                else:
                    t_ = [truth_normal[k] for k in idx]
                    p_ = [drop_on(preds[k], ignore[k]) for k in idx]
                print(f"{name:14s}", score_line(t_, p_))

    if show:
        for k, f in enumerate(files):
            fig, axs = plt.subplots(1, len(results) + 1, figsize=(6 * (len(results) + 1), 6))
            show_outlines(images[k], truth_normal[k], axs[0], color="lime", title=f"{modes[k]}: normal (green)")
            for o in utils.outlines_list(np.where(ignore[k], truth_all[k], 0)):
                axs[0].plot(o[:, 0], o[:, 1], color="orange", linewidth=1.5)
            axs[0].set_title(f"{modes[k]}: your ROIs (abn_/unsure_ orange)")
            for ax, name in zip(axs[1:], results):
                show_outlines(images[k], results[name][k], ax, title=name)
            plt.show()
    return results

In [ ]:
COMPARE = ["pretrained", "GRAMW_v2", "GRAMW_v3", "GRAMW_v4"]
results = evaluate(COMPARE, show=True)